<a href="https://colab.research.google.com/github/hutengdai/hutengdai.github.io/blob/master/LING345_python_tutorial_answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A minimalistic Python tutorial &mdash; ANSWER KEY

**LING 345**

Every blank is filled in. Run the cells from top to bottom and all three exercises come out green.

Each exercise is followed by a **Why this answer** block explaining the code and the near-misses the checker watches for.

The student copy is `LING345_python_tutorial.ipynb`.

## Before you start: how a notebook works

A notebook is a stack of **cells**.

* **To run a cell**,  press <kbd>Shift</kbd>+<kbd>Enter</kbd>, or click the
  &#9654; button on its left.
* Cells in a notebook share one memory. A variable you make in one cell is still there in the next.
* **Order matters.** Run the cells top to bottom the first time. If Python says something
  is `not defined`, you almost certainly skipped a cell above.
* **Editing a cell changes nothing until you run it again.** This is the number one
  source of "but I fixed it!" moments.
* You can edit and re-run any cell as often as you like, learning through trials and errors.
* If you want a hard reset, do **Runtime &rarr; Restart session and run all** in Colab.
* You should copy this notebook to your own google drive for editing.

**Colab users:** the first time you edit anything, Colab offers to *Save a copy in Drive*. Say yes, or your work is not saved anywhere.

Run the next cell. It loads the self-check.

In [ ]:
#@title Run me first: the self-check (click the play button)
# =====================================================================
#  LING 345 -- the self-check.  Run this cell once, at the top.
#
#  You never need to change anything in here, and nothing is hidden
#  from you either: it is ordinary Python and you are welcome to read
#  it.  It gives you four things:
#
#      check_1(), check_2(), check_3()   mark one exercise
#      hint(1), hint(2), hint(3)         a nudge, never the answer
#      report()                          everything at once, at the end
#      show_wave(), feature_table()      pictures, used later on
# =====================================================================
import inspect
import math

import matplotlib.pyplot as plt


# --- finding your variables ------------------------------------------
# Your answers live in the notebook's own memory, not in here, so the
# checker goes and looks for them.  If it ever cannot find them, you
# can always be explicit:   check_1(globals())

def _notebook_names(given=None):
    if isinstance(given, dict):
        return given
    space = {}
    try:                                    # IPython's copy, as a backup
        from IPython import get_ipython
        user_ns = getattr(get_ipython(), "user_ns", None)
        if isinstance(user_ns, dict):
            space.update(user_ns)
    except Exception:
        pass
    frame = inspect.currentframe()          # step out of this file's own frames
    steps = 0
    while frame is not None and steps < 20:
        if frame.f_globals.get("__name__") == "__main__" and "_notebook_names" not in frame.f_locals:
            space.update(frame.f_globals)
            space.update(frame.f_locals)
            break
        frame = frame.f_back
        steps += 1
    return space


# --- what counts as an answer ----------------------------------------

def _answered(value, depth=0):
    """A leftover ... is not an answer, and neither is a list holding one."""
    if value is Ellipsis:
        return False
    if depth < 3 and isinstance(value, (list, tuple)):
        return all(_answered(x, depth + 1) for x in value)
    return True


def _same(got, want, exact_type=False):
    if isinstance(got, bool) != isinstance(want, bool):
        return False                        # True is not 1 in this course
    if exact_type and isinstance(want, float) and not isinstance(got, float):
        return False                        # 0 from // is not 0.000375 from /
    if isinstance(got, float) or isinstance(want, float):
        try:
            return math.isclose(got, want, rel_tol=1e-9, abs_tol=1e-12)
        except TypeError:
            return False
    return got == want


def _short(value):
    text = repr(value)
    return text if len(text) <= 34 else text[:31] + "..."


# --- the checker itself ----------------------------------------------
_SECTION_TITLES = {1: "Numbers, text, lists, and dictionaries",
                   2: "Making a decision",
                   3: "Your first speech recognizer"}
_SECTION_SCORES = {}


def _mark(number, wanted, hints=(), exact_type=(), needs=(), given=None):
    here = _notebook_names(given)
    missing = [n for n in needs if n not in here]
    if missing:
        print("I cannot find: " + ", ".join(missing))
        print("You have probably skipped a cell, or edited one without running it.")
        print("Fix: Runtime > Restart session and run all   (Colab)")
        print("     Kernel > Restart Kernel and Run All Cells   (Jupyter)")
        return

    rows, right = [], 0
    for name, want in wanted:
        got = here.get(name, Ellipsis)
        blank = name not in here or not _answered(got)
        ok = (not blank) and _same(got, want, name in exact_type)
        right += 1 if ok else 0
        rows.append((name, got, ok, blank))

    width = max(len(r[0]) for r in rows)
    print("Exercise %d -- %s" % (number, _SECTION_TITLES[number]))
    print("%d of %d right" % (right, len(rows)))
    print()
    print("   %-*s  %-34s  %s" % (width, "variable", "your value", "evaluation"))
    print("   " + "-" * (width + 48))
    for name, got, ok, blank in rows:
        evaluation = "correct!" if ok else ("not filled in yet" if blank else "not right yet")
        shown = "(still a blank)" if blank else _short(got)
        print("   %-*s  %-34s  %s" % (width, name, shown, evaluation))

    notes = []
    for name, got, ok, blank in rows:
        if ok or blank:
            continue
        for test, message in hints:
            try:
                hit = test(name, got)
            except Exception:
                hit = False
            if hit:
                notes.append((name, message))
                break
    if notes:
        print()
        print("   what to look at:")
        for name, message in notes:
            for i, line in enumerate(message.split("\n")):
                print("     %s %s" % (name + ":" if i == 0 else " " * (len(name) + 1), line))

    print()
    if right == len(rows):
        print("   All of them are right. Move on.")
    else:
        print("   Fix the cell above, run it again, and this check runs with it.")
        print("   Still stuck? Run  hint(%d)  in the next cell." % number)
    _SECTION_SCORES[number] = (right, len(rows))


# --- graduated hints, which never give the answer away ----------------
_HINTS = {
    1: ["Everything you need is in the demo cell just above the exercise.\n"
        "Every one of the five answers appears there in some form.",
        "num_samples counts.  last_sample uses a negative number in the\n"
        "brackets.  first_three uses a colon.  duration divides, and it has\n"
        "to keep the decimal part.  no_frequency looks a key up in the table."],
    2: ["Two things are being tested: the ORDER of the two conditions, and\n"
        "whether you wrote > or >=.",
        "Ask the narrow question first (is it above 0.5?), then the wide one\n"
        "(is it above 0.05?).  And 'above 0.05' does not include 0.05 itself."],
    3: ["The loop keeps updating the best score and the best, most plausible\n"
        "word for a new recording.  A recognizer returns a word.",
        "Note that the return indentation should be carefully placed so it is\n"
        "not inside the loop."],
}
_HINTS_GIVEN = {}


def hint(number):
    """A nudge for one exercise. Run it again for a bigger one."""
    level = _HINTS_GIVEN.get(number, 0)
    messages = _HINTS[number]
    print(messages[min(level, len(messages) - 1)])
    if level + 1 < len(messages):
        print()
        print("(run hint(%d) again for a bigger hint)" % number)
    _HINTS_GIVEN[number] = level + 1


def report():
    """Everything at once. Run this at the end, before you hand the notebook in."""
    right = total = 0
    print("=" * 64)
    print("LING 345 -- Python self-check -- your report")
    print("=" * 64)
    for number in (1, 2, 3):
        title = _SECTION_TITLES[number]
        if number in _SECTION_SCORES:
            r, n = _SECTION_SCORES[number]
            verdict = "solid" if r == n else ("nearly" if r >= n - 1 else "keep going")
            print("   Exercise %d  %-38s  %d/%d  %s" % (number, title, r, n, verdict))
            right += r
            total += n
        else:
            print("   Exercise %d  %-38s   not run yet" % (number, title))
    print("-" * 64)
    print("   TOTAL: %d of %d" % (right, total))
    print()
    if total == 12 and right == 12:
        print("   All green. Follow the hand-in instructions at the bottom.")
    else:
        print("   Anything not green: fix that exercise cell, run it again (the")
        print("   check runs with it), then run report() once more.")
    print("=" * 64)


# --- section 4 runs on the recognizer you wrote in Exercise 3 ----------

def exercise_3_done():
    """True once recognize() actually hands back a word."""
    here = _notebook_names()
    if not callable(here.get("recognize")) or "model" not in here:
        print("Section 4 uses the recognize() you write in Exercise 3.")
        print("Go back and finish Exercise 3 first, then run this cell again.")
        return False
    try:
        answer = here["recognize"](here["demo_no"], here["model"])
    except Exception as problem:
        print("Exercise 3 is not working yet: %s" % problem)
        print("Fix that cell, run it, then come back here.")
        return False
    if not isinstance(answer, str):
        print("Exercise 3 is not finished: recognize() is handing back")
        print("%r instead of a word. Fill in its last line, run that cell," % (answer,))
        print("then come back here.")
        return False
    return True


# --- pictures ---------------------------------------------------------

def show_wave(recording, title):
    """Draw one recording, with a red dot at every zero crossing."""
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.axhline(0, color="0.4", linewidth=1)
    ax.plot(range(len(recording)), recording, color="#1b1b1b",
            linewidth=1.4, marker="o", markersize=3.5)
    crossings = [i for i in range(1, len(recording))
                 if recording[i - 1] * recording[i] < 0]
    ax.plot([i - 0.5 for i in crossings], [0] * len(crossings), "o",
            color="#b3261e", markersize=8, zorder=3,
            label="zero crossing (%d in all)" % len(crossings))
    size = len(recording) // 8
    for k in range(1, 8):
        ax.axvline(k * size - 0.5, color="#7c3aed", linestyle="--",
                   linewidth=1, alpha=.5)
    top = max(abs(v) for v in recording)
    for k in range(8):
        ax.text(k * size + size / 2 - 0.5, top * 1.25, str(k),
                ha="center", color="#7c3aed", fontsize=9)
    ax.set_title('"%s"' % title)
    ax.set_xlabel("sample number")
    ax.set_ylabel("level")
    ax.set_ylim(-top * 1.5, top * 1.5)
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    plt.show()


def feature_table(recording, name=""):
    """Show the work inside features(): every slice, its raw rate, its share."""
    pieces = slices(recording, 8)
    raw = [zero_crossings(p) for p in pieces]
    total = sum(raw)
    print("features(%s), one row per slice" % name)
    print()
    print("   %5s | %-26s | %7s | %7s" % ("slice", "the samples in it", "raw", "share"))
    print("   " + "-" * 56)
    for i, piece in enumerate(pieces):
        share = round(raw[i] / total, 3) if total else 0.0
        print("   %5d | %-26s | %7s | %7s" % (i, str(piece), raw[i], share))
    print("   " + "-" * 56)
    print("   %5s | %-26s | %7s | %7s" % ("total", "", round(total, 3), 1.0))

# --- the three exercises' answers -------------------------------------

def check_1(given=None):
    """Mark Exercise 1."""
    _mark(1,
        wanted=[("num_samples", 6),
                ("last_sample", 0.1),
                ("first_three", [0.0, 0.3, -0.7]),
                ("duration", 0.000375),
                ("no_frequency", 4)],
        exact_type=("duration",),
        given=given,
        hints=[
          (lambda n, v: n == "last_sample" and v == 0.0,
           "that is samples[0], the FIRST one. A negative index counts\n"
           "back from the end."),
          (lambda n, v: n == "first_three" and v == [0.0, 0.3],
           "samples[:3] takes three, not two. Python stops BEFORE the\n"
           "number you write, so :3 gives you positions 0, 1 and 2."),
          (lambda n, v: n == "first_three" and v == [0.0, 0.3, -0.7, 0.9],
           "samples[:3] stops before position 3, so it gives you three,\n"
           "not four."),
          (lambda n, v: n == "duration" and v == 0,
           "you used // , which throws the decimal part away. Six samples\n"
           "at 16000 a second really is a tiny fraction of a second, and\n"
           "only a single / lets you see it."),
          (lambda n, v: n == "no_frequency" and v == 2,
           'that is the count for "go". Look at which key you asked for.'),
          (lambda n, v: n == "no_frequency" and v == 0,
           'a 0 means the key you asked for was not in the table. "no" IS\n'
           "in there, so ask for it by name."),
        ])


def check_2(given=None):
    """Mark Exercise 2."""
    _mark(2,
        wanted=[("label_loud", "loud"),
                ("label_speech", "speech"),
                ("label_quiet", "silence"),
                ("label_edge", "silence")],
        needs=("describe",),
        given=given,
        hints=[
          (lambda n, v: n == "label_loud" and v == "speech",
           "the wide test ran first, so the narrow one below it can never\n"
           "be reached. Put  > 0.5  above  > 0.05 ."),
          (lambda n, v: n == "label_edge" and v == "speech",
           "you used >= somewhere. Exactly 0.05 is not ABOVE 0.05, so it\n"
           'has to fall through to "silence".'),
          (lambda n, v: v is None,
           "describe() handed back None, which means some path through it\n"
           "reaches no return at all. Every branch needs one."),
        ])


def check_3(given=None):
    """Mark Exercise 3."""
    _mark(3,
        wanted=[("distance_to_no", 0.054343),
                ("answer", "go"),
                ("answer_self", "no")],
        needs=("features", "distance", "recognize", "model", "demo_no", "demo_mystery"),
        given=given,
        hints=[
          (lambda n, v: n == "answer" and v == "no",
           'it answered "no", which is the FIRST word the loop looks at.\n'
           "That happens when the return is indented inside the for loop:\n"
           "the function then quits on word one and never sees the rest.\n"
           "Move it left, so it lines up with  best_word = None  and runs\n"
           "after the loop has finished."),
          (lambda n, v: n == "answer" and v is None,
           "it never picked a word: best_word was still None at the end.\n"
           "Hand back the variable the loop filled in."),
          (lambda n, v: n == "answer" and isinstance(v, float),
           "that is the winning distance, not the winning word. The score\n"
           "lives in best_score; the word that earned it is in best_word."),
          (lambda n, v: n == "answer" and not isinstance(v, str),
           "recognize() has to hand back a word, as text. Return the\n"
           "variable holding the winning word."),
          (lambda n, v: n == "answer_self" and v == "go",
           "a stored recording has to recognize itself. Its distance from\n"
           "its own entry in the model is exactly 0, and nothing can\n"
           "beat that."),
        ])


print("Self-check ready, let's goooooooo!")

## 1. Numbers, text, lists, and dictionaries

`int`, `float`, `str`, `list`, `dict`

A recording is just a list of numbers. Each one is a **sample** of the amplitude of the continuous waveform.

In [ ]:
samples = [0.0, 0.3, -0.7, 0.9, -0.2, 0.1]
sample_rate = 16000             # int: samples per second
word = "no"                     # str: text goes in quotes

print("how many samples: ", len(samples))
print("the first one:    ", samples[0])
print("the last one:     ", samples[-1])
print("the first three:  ", samples[:3])
print("how long, seconds:", len(samples) / sample_rate)

# a dictionary is a lookup table:
frequency = {"no": 4, "go": 2}
print("the frequency of 'no':", frequency["no"])

# .get(key, 0) hands back a fallback 0 instead of an error when the key is
# missing from the dictionary -- so a missing word gives 0, not a crash
print("the frequency of 'up':", frequency.get("up", 0))

Two things worth keeping:

* `samples[-1]` counts from the **end**, so `-1` is the last item and `-2` the one before it.

* Python has **two** division signs, and they are not the same. `7 / 2` is `3.5`, keeping the
  decimal part---so called regular division. `7 // 2` is `3`, also known as floor division. You will need to choose between them below.

### Exercise 1

Replace every `...` below with Python, then run the cell. The check at the bottom runs with it.

In [ ]:
samples = [0.0, 0.3, -0.7, 0.9, -0.2, 0.1]
sample_rate = 16000

num_samples = len(samples)             # how many samples there are
last_sample = samples[-1]              # the last sample, WITHOUT using len()
first_three = samples[:3]              # a list of the first three samples
duration = len(samples) / sample_rate  # how long the recording lasts, in seconds.
                                       # This one must come out as a decimal, not a
                                       # whole number, so think about which kind of
                                       # division.

frequency = {"no": 4, "go": 2}
no_frequency = frequency["no"]         # how many times "no" was said, looked up in the table above


check_1()

<details>
<summary><b>Why this answer</b> &mdash; click to open</summary>

`num_samples = len(samples)` &mdash; `len()` counts the items in a list, so `6`.

`last_sample = samples[-1]` &mdash; a negative index counts back from the end. `samples[len(samples) - 1]` gives the same `0.1`, but the exercise asks for the version that does not need `len()`.

`first_three = samples[:3]` &mdash; a slice stops *before* the number you write, so `:3` gives positions 0, 1 and 2. Off by one either way is the usual slip: `[:2]` gives two, `[:4]` gives four.

`duration = len(samples) / sample_rate` &mdash; regular division, one slash. `/` keeps the decimal and gives `0.000375`; `//` floors it to `0`. Six samples at 16000 Hz really is a tiny fraction of a second, and only `/` shows it.

`no_frequency = frequency["no"]` &mdash; square brackets look a key up, so `4`. `frequency.get("no", 0)` works too, since `"no"` is in the table. A `2` means you asked for `"go"`; a `0` means you used `.get()` on a key the table does not have and got the fallback.

</details>

## 2. Making a decision

`if` / `elif` / `else`, and conditions

The `if` statement is how Python makes decisions. Here we use it to classify a recording as
`"loud"`, `"speech"` or `"silence"` from its **peak level**: what's the maximum loudness of each recording.

Indentation matters. After a line that ends in a colon, the next line is indented by four
spaces &mdash; or you can just press <kbd>Tab</kbd>.

The code below is written as a **function**: a piece of code with a name after `def`, which takes
something in (also called arguments), such as `level` here, then outputs something with `return`.

Once we define the function, you can use it as often as you like.

In [ ]:
def describe(level):
    if level > 0.5:
        return "loud"
    elif level > 0.05:
        return "speech"
    else:
        return "silence"


for x in [0.9, 0.2, 0.01]:
    print(x, "->", describe(x))

### Exercise 2

Fill in the three conditions and the three answers. The order of the tests is the exercise.

In [ ]:
def describe(level):
    """Three answers: "loud" above 0.5, "speech" above 0.05,
    otherwise "silence".
    """
    if level > 0.5:
        return "loud"
    elif level > 0.05:
        return "speech"
    else:
        return "silence"


# do not change these four lines
label_loud = describe(0.9)
label_speech = describe(0.2)
label_quiet = describe(0.01)
label_edge = describe(0.05)


check_2()

<details>
<summary><b>Why this answer</b> &mdash; click to open</summary>

**Order is the exercise.** `0.9` is above `0.05` as well as above `0.5`, so putting `> 0.05` first swallows every loud level and `"loud"` becomes unreachable. Narrow test first, wide test second.

**`>` and not `>=`.** The docstring says *above* 0.05, and `describe(0.05)` is checked on purpose: 0.05 is not above 0.05, so it falls through to `"silence"`. `>=` makes that one case return `"speech"`, and it is the only test that catches it.

**The `else` takes no condition.** Everything that failed both tests lands there, which is the silent case.

Expected: `label_loud` &rarr; `"loud"`, `label_speech` &rarr; `"speech"`, `label_quiet` &rarr; `"silence"`, `label_edge` &rarr; `"silence"`.

</details>

## 3. Your first speech recognizer

everything above, in one small program

Now let's build our first speech recognizer.

Every speech recognizer, from this one to the one in your phone, has the same three jobs:

| | job | when it happens |
|---|---|---|
| **1** | **Data processing.** Turn a recording into a handful of numbers that describe it. | training *and* testing |
| **2** | **Training.** Work out what each word looks like in those numbers. | once |
| **3** | **Testing.** Score a new recording against every word, and keep the best. | every new recording |

Ours cuts each recording into eight slices and turns each slice into one number: how often the waveform crosses zero there. That is what separates **n** from **g**. [g] is a burst and flips sign on nearly every sample; [n] is a slow hum that barely crosses at all. Both words end in the same vowel, so the difference has to be at the onset.

First, imagine we have recordings of "no" averaged into `demo_no`, and recordings of "go" averaged into `demo_go`. Now we have a new recording stored in `demo_mystery`, and our job is to recognize whether it is a no or a go.

In [ ]:
# Three demo recordings, typed out by hand rather than recorded, so that the
# numbers below come out the same for everybody.  Your own voice goes through
# this very same code later on.

demo_no = [0.5, 0.5, 0.5, 0.5, -0.5, -0.5, -0.5, -0.5,    # n: one slow swing, up then down
           0.8, 0.8, -0.8, -0.8, 0.8, 0.8, -0.8, -0.8,    # then the vowel: a faster wave
           0.8, 0.8, -0.8, -0.8, 0.8, 0.8, -0.8, -0.8]    # 24 samples in all

demo_go = [0.5, -0.5, 0.5, -0.5, 0.5, -0.5, 0.5, -0.5,    # g: a burst, up down up down
           0.8, 0.8, -0.8, -0.8, 0.8, 0.8, -0.8, -0.8,    # then the very same vowel
           0.8, 0.8, -0.8, -0.8, 0.8, 0.8, -0.8, -0.8]    # 24 samples in all

demo_mystery = [0.5, -0.5, 0.5, -0.5, 0.5, -0.5,              # a burst again, a shorter one
                0.8, 0.8, -0.8, -0.8, 0.8, 0.8, -0.8, -0.8,   # and the vowel
                0.8, 0.8, -0.8, -0.8, 0.8, 0.8, -0.8, -0.8,
                0.8, 0.8]                                     # 24 samples, so which word is it?

print("no     ", len(demo_no), "samples")
print("go     ", len(demo_go), "samples")
print("mystery", len(demo_mystery), "samples")

### Look at the two words first

The dashed purple lines mark the eight slices. Every red dot is a zero crossing. Count the dots at the left of each picture: that difference is the recognizer.

In [ ]:
show_wave(demo_no, "no")
show_wave(demo_go, "go")

### The four functions the recognizer is built from

A function can be reused over and over. It takes something in (the input), and it hands something back with `return`. Now let's write some functions as the core components of our recognizer.

Each of the next four cells defines one function, and each is followed by a **try it** cell. Run it, change the numbers, run it again.

A cell containing only a `def` prints nothing. That is normal: Python has learned the function and is waiting for you to call it.

#### Function 1 of 4: `zero_crossings`

I essentially wrote a function called `zero_crossings` to count how often the wave crosses zero. The idea is that [g] crosses the zero more often than [n].

Walk through the recording one sample at a time. If a sample and the one before it have opposite signs, the waveform crossed zero between them. Multiplying two numbers of opposite sign gives a negative, which is what `recording[i - 1] * recording[i] < 0` tests.

Dividing by the length turns the count into a **rate**, so recordings of different lengths stay comparable.

In [ ]:
def zero_crossings(recording):
    """How often the wave changes sign, per sample."""
    if len(recording) < 2:                       # too short to cross anything
        return 0.0
    crossings = 0                                # nothing counted yet
    for i in range(1, len(recording)):           # every sample except the first
        if recording[i - 1] * recording[i] < 0:  # one up and the next down: a crossing
            crossings = crossings + 1            # count it
    return round(crossings / len(recording), 3)  # per sample, so length does not matter

In [ ]:
# Try it.  Change these lists and run the cell again -- that is the whole point
# of a notebook.  Which one wiggles more?

print("a slow wave  [1, 1, -1, -1] ->", zero_crossings([1, 1, -1, -1]))
print("a fast wave  [1, -1, 1, -1] ->", zero_crossings([1, -1, 1, -1]))
print("flat silence [0, 0,  0,  0] ->", zero_crossings([0, 0, 0, 0]))
print()
print("the whole word 'no' ->", zero_crossings(demo_no))
print("the whole word 'go' ->", zero_crossings(demo_go))

#### Function 2 of 4: `slices`

One number for a whole word throws away *where* the wiggling happened, and that is the one thing separating **no** from **go**. So cut the recording into equal pieces and describe each piece separately.

In [ ]:
def slices(recording, how_many):
    """Cut a recording into `how_many` equal pieces, in order."""
    size = len(recording) // how_many                        # how many samples in each piece
    pieces = []                                              # collect them here
    for i in range(how_many):                                # one piece at a time
        pieces.append(recording[i * size:(i + 1) * size])    # from here up to there
    return pieces                                            # the pieces, in order

In [ ]:
# Try it.  Cut a short list into 4 pieces so you can see the shape of the answer.

for piece in slices([1, 2, 3, 4, 5, 6, 7, 8], 4):
    print(piece)

print()
print("the word 'no', cut into 8 pieces:")
for piece in slices(demo_no, 8):
    print(piece)

#### Function 3 of 4: `features`

Cut into 8 pieces, measure each piece, then divide every number by the total. What gets compared is the **shape** of the word rather than the overall rate at which it was said.

Loudness never came into it: counting sign changes ignores amplitude, so turning the volume up or down leaves these numbers unchanged.

We call this kind of information **features**, and we store them with the function `features` into a list of values. Eight numbers describing one recording is what everyone in speech technology calls a **feature vector**.

In [ ]:
def features(recording):
    """Describe one recording with 8 numbers: how fast it wiggles, start to end."""
    numbers = []                                     # start with none
    for piece in slices(recording, 8):               # walk the word from start to end
        numbers.append(zero_crossings(piece))        # how fast the wave wiggles here
    total = sum(numbers)                             # one voice can wiggle more than another
    if total == 0:                                   # a silent recording: nothing to scale
        return numbers
    shares = []                                      # each slice's share of the total
    for n in numbers:
        shares.append(round(n / total, 3))           # so compare the shape, not the size
    return shares

In [ ]:
# Try it.  These three lines are the heart of the recognizer -- read them before
# you go on.  The words differ at the START, which is where n and g live.

print("no      ->", features(demo_no))
print("go      ->", features(demo_go))
print("mystery ->", features(demo_mystery))

The same calculation with the work written out, one row per slice.

In [ ]:
feature_table(demo_go, "demo_go")

# TRY: change demo_go to demo_no above and run this again.

#### Function 4 of 4: `distance`

Two feature vectors, subtracted position by position. Each difference is squared, so `-0.2` counts as much as `+0.2` and the two cannot cancel. A **small** total means the two recordings look alike.

In [ ]:
def distance(a, b):
    """How far apart two lists of eight numbers are.  Small = they look alike."""
    total = 0.0                                  # nothing added up yet
    for i in range(len(a)):                      # a[0] against b[0], and so on
        total = total + (a[i] - b[i]) ** 2       # squared, so + and - cannot cancel
    return total                                 # a small total means the two look alike

In [ ]:
# Try it.  A thing is always distance 0 from itself.

print("no      vs no  ->", distance(features(demo_no), features(demo_no)))
print("no      vs go  ->", distance(features(demo_no), features(demo_go)))
print()
print("mystery vs no  ->", distance(features(demo_mystery), features(demo_no)))
print("mystery vs go  ->", distance(features(demo_mystery), features(demo_go)))
print()
print("Which of the last two is smaller?  That is the answer the recognizer has to give.")

#### The decision, drawn

Look at the first two slices. `"no"` starts at `0.0`: its onset never crosses zero. `"go"` and `"mystery"` both start high. The two distances printed underneath are the decision.

In [ ]:
# A picture of what the recognizer actually hears.  Nothing to fill in here --
# just run it.  Top row: the raw recordings.  Bottom row: the 8 feature numbers.

import matplotlib.pyplot as plt

recordings = [("no", demo_no), ("go", demo_go), ("mystery", demo_mystery)]
fig, axes = plt.subplots(2, 3, figsize=(11, 4.6))

for col, (name, rec) in enumerate(recordings):
    axes[0][col].plot(rec, marker=".")
    axes[0][col].axhline(0, color="grey", linewidth=.8)
    axes[0][col].set_title('"%s" -- the wave' % name)
    axes[0][col].set_ylim(-1, 1)

    axes[1][col].bar(range(8), features(rec))
    axes[1][col].set_title('"%s" -- the 8 features' % name)
    axes[1][col].set_xlabel("slice, start to end")
    axes[1][col].set_ylim(0, .25)

fig.tight_layout()
plt.show()

print("distance from the mystery recording to 'no':", distance(features(demo_mystery), features(demo_no)))
print("distance from the mystery recording to 'go':", distance(features(demo_mystery), features(demo_go)))
print("Smaller means more alike, so the answer has to be 'go'.")

#### One notebook trick

`help(some_function)` prints what a function is for. Try `help(len)`, `help(round)`, anything. Typing `features??` in a cell shows the source code itself.

In [ ]:
help(features)

### Picking the winner

Step 3 goes through the words one at a time and holds on to the best one so far. Run this to watch that idea on its own, before you meet it inside the recognizer.

In [ ]:
scores = {"no": 0.42, "go": 0.07, "up": 0.31}   # one distance per word

best_word = None              # nothing chosen yet
best_score = float("inf")     # bigger than every real number

for word in scores:                    # one word at a time
    print("looking at", word, scores[word])
    if scores[word] < best_score:      # closer than the best so far?
        best_word = word               # then this is the new best
        best_score = scores[word]      # and this is the score to beat now

print()
print("the closest word:", best_word)

`float("inf")` is infinity. Every real number is below it, so the first word always wins and leaves the rest something to beat. Starting at `0` would break this, because a distance is never below zero, so no word could ever win.

Change the numbers in `scores` above, add a fourth word, run it again.

### Exercise 3

Everything is written except the last line of `recognize`. Fill it in and run the cell.

In [ ]:
# STEP 2, TRAINING.  The model is everything the machine knows about each
# word, and ours is just those 8 numbers.

model = {"no": features(demo_no), "go": features(demo_go)}   # one entry per word


# STEP 3, TESTING.  The loop below has the same shape as the demo above.

def recognize(recording, model):
    # do not change these two lines
    best_word = None             # nothing chosen yet
    best_score = float("inf")    # bigger than every real distance

    for word in model:           # "no", then "go"
        score = distance(features(recording), model[word])
        if score < best_score:   # closer than the best so far?
            best_word = word     # then this is the new best
            best_score = score   # and this is the score to beat now

    return best_word             # hand back whichever word ended up closest


# do not change these three lines
distance_to_no = distance(features(demo_mystery), model["no"])
answer = recognize(demo_mystery, model)
answer_self = recognize(demo_no, model)

print("distance from the mystery recording to 'no':", distance_to_no)
print("the recognizer says the mystery word is:    ", answer)
print("and 'no' recognizes itself as:              ", answer_self)

check_3()

<details>
<summary><b>Why this answer</b> &mdash; click to open</summary>

One blank: `return best_word`.

**`best_word`, not `best_score`.** The loop keeps two things in step, the smallest distance so far and the word that earned it, and a recognizer returns the *word*. `best_score` answers `0.007777` instead of `"go"`, and the checker rejects it because a number is not a word.

**Return it *after* the loop**, at the function's own indentation. A `return` inside the `for` body quits on the first word it looks at, which here is always `"no"`.

**Why the answer is `"go"`:** the mystery recording opens with a burst that changes sign on nearly every sample, which is exactly what `features()` measures. Its first two slices come out at `0.2`, close to `"go"`'s `0.182` and far from `"no"`, which opens at `0.0`. The distances follow: `0.007777` to `"go"` against `0.054343` to `"no"`, and the smaller wins.

`answer_self` is the sanity check: `demo_no` scored against a model built from `demo_no` has distance exactly `0`, so a working recognizer has to call it `"no"`.

</details>

## 4. Now with real sound

The three exercises are done. Everything from here is ungraded.

Your recognizer does not care where its numbers come from. So far they were typed by hand; now we feed it real audio, three ways, each a fallback for the one before:

1. **A synthesized word.** Always works, no permissions needed.
2. **Your own microphone.** Works in Colab.
3. **A sound file you upload.** For when the microphone will not cooperate.

The next cell loads the audio helper, which turns sound into a list of numbers between -1.0 and 1.0.

In [ ]:
#@title Run me: the audio helper (click the play button)
# =====================================================================
#  LING 345 -- the audio helper.  Run this cell once.  It is plumbing:
#  it turns sound into a list of numbers between -1.0 and 1.0, which is
#  the only thing your recognizer knows how to read.
#
#      samples = record(2.0)          record 2 seconds from your microphone
#      samples = demo_audio("no")     a stand-in sound, no microphone needed
#      samples = upload_audio()       use a sound file instead
#      samples = trim_silence(samples)  throw away the quiet ends
#
#  You are welcome to read it, but you do not need to.
# =====================================================================
import array, base64, json, math, struct, subprocess, sys

SAMPLE_RATE = 16000


# ------------------------------------------------------------------ setup
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


class MicrophoneError(RuntimeError):
    """Raised with a plain-English explanation when recording is impossible."""


_HELP = """
--------------------------------------------------------------------
We could not record from your microphone.  Two things you can try:

  1. UPLOAD A SOUND FILE INSTEAD
       samples = upload_audio()     # then pick a file from your computer
     (Record a voice memo on your phone, email it to yourself, and
      choose that file.  Any format works in Colab; use .wav locally.)

  2. USE THE STAND-IN SOUND so you can keep going with the lesson
       samples = demo_audio("no")   # or demo_audio("go")

If you want to fix the microphone: make sure you clicked "Allow" on the
browser's permission popup, that no other app (Zoom, Teams) is holding
the mic, and that the notebook tab is the one you are looking at.
--------------------------------------------------------------------
"""


# ------------------------------------------------------- number crunching
def _b64_pcm16_to_floats(b64_string):
    """base64 of little-endian 16-bit PCM -> list of floats in -1..1"""
    raw = base64.b64decode(b64_string)
    raw = raw[: len(raw) - (len(raw) % 2)]
    a = array.array("h")
    a.frombytes(raw)
    if sys.byteorder == "big":
        a.byteswap()
    return [v / 32768.0 for v in a]


def _resample(samples, src_rate, dst_rate):
    """Convert a list of floats from src_rate to dst_rate samples/second."""
    src_rate, dst_rate = int(src_rate), int(dst_rate)
    if not samples or src_rate == dst_rate:
        return list(samples)
    n_out = max(1, int(round(len(samples) * dst_rate / float(src_rate))))
    try:                                    # best: scipy (Colab + Anaconda have it)
        from fractions import Fraction
        import numpy as np
        from scipy.signal import resample_poly
        f = Fraction(dst_rate, src_rate).limit_denominator(1000)
        y = resample_poly(np.asarray(samples, dtype=np.float64), f.numerator, f.denominator)
        return [float(v) for v in y]
    except Exception:
        pass
    try:                                    # ok: numpy only
        import numpy as np
        x = np.asarray(samples, dtype=np.float64)
        if src_rate > dst_rate:
            w = int(round(src_rate / float(dst_rate)))
            if w > 1:
                x = np.convolve(x, np.ones(w) / w, mode="same")
        return [float(v) for v in np.interp(np.linspace(0, len(x) - 1, n_out),
                                            np.arange(len(x)), x)]
    except Exception:
        pass
    x = list(samples)                       # last resort: pure Python
    if src_rate > dst_rate:
        w = int(round(src_rate / float(dst_rate)))
        if w > 1:
            half, n, sm = w // 2, len(x), []
            for i in range(n):
                lo, hi = max(0, i - half), min(n, i - half + w)
                sm.append(sum(x[lo:hi]) / (hi - lo))
            x = sm
    n, out = len(x), []
    for k in range(n_out):
        p = k * (n - 1) / float(n_out - 1) if n_out > 1 else 0.0
        i = int(p)
        if i >= n - 1:
            out.append(x[-1])
        else:
            out.append(x[i] * (1 - (p - i)) + x[i + 1] * (p - i))
    return out


def _finish(samples, src_rate, sample_rate, seconds=None, normalise=True):
    """Resample, trim/pad to the requested length, and gently normalise."""
    out = _resample(samples, src_rate, sample_rate)
    if seconds:
        want = int(round(seconds * sample_rate))
        out = out[:want] + [0.0] * max(0, want - len(out))
    if normalise:
        peak = max((abs(v) for v in out), default=0.0)
        if peak > 1e-6:
            out = [v * (0.95 / peak) for v in out]
    return out


# ------------------------------------------- read a .wav file, stdlib only
def wav_bytes_to_samples(data, sample_rate=SAMPLE_RATE):
    """RIFF/WAVE bytes -> (mono list of floats, sample_rate).  No dependencies.
    Handles 8/16/24/32-bit integer PCM and 32/64-bit float PCM, any channel count."""
    if len(data) < 12 or data[0:4] != b"RIFF" or data[8:12] != b"WAVE":
        raise ValueError("That file is not a .wav file. Please export/convert it to WAV.")
    fmt_tag = chans = rate = bits = frames = None
    pos = 12
    while pos + 8 <= len(data):
        cid = data[pos:pos + 4]
        size = struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt " and len(body) >= 16:
            fmt_tag, chans, rate, _br, _ba, bits = struct.unpack("<HHIIHH", body[:16])
            if fmt_tag == 0xFFFE and len(body) >= 40:          # WAVE_FORMAT_EXTENSIBLE
                fmt_tag = struct.unpack("<H", body[24:26])[0]
        elif cid == b"data":
            frames = body
        pos += 8 + size + (size & 1)
    if fmt_tag is None or frames is None:
        raise ValueError("This .wav file is damaged (no 'fmt ' or 'data' chunk).")
    if fmt_tag not in (1, 3):
        raise ValueError("This WAV uses a compressed encoding. Please export plain PCM WAV.")
    chans = max(1, chans or 1)

    if fmt_tag == 3 and bits == 32:
        n = len(frames) // 4
        flat = list(struct.unpack("<%df" % n, frames[:4 * n]))
    elif fmt_tag == 3 and bits == 64:
        n = len(frames) // 8
        flat = list(struct.unpack("<%dd" % n, frames[:8 * n]))
    elif bits == 8:
        flat = [(b - 128) / 128.0 for b in frames]
    elif bits == 16:
        a = array.array("h"); a.frombytes(frames[: len(frames) - (len(frames) % 2)])
        if sys.byteorder == "big": a.byteswap()
        flat = [v / 32768.0 for v in a]
    elif bits == 24:
        flat = []
        for i in range(0, len(frames) - 2, 3):
            v = frames[i] | (frames[i + 1] << 8) | (frames[i + 2] << 16)
            flat.append((v - 0x1000000 if v & 0x800000 else v) / 8388608.0)
    elif bits == 32:
        a = array.array("i"); a.frombytes(frames[: len(frames) - (len(frames) % 4)])
        if sys.byteorder == "big": a.byteswap()
        flat = [v / 2147483648.0 for v in a]
    else:
        raise ValueError("Unsupported WAV bit depth: %s" % bits)

    if chans > 1:
        n = len(flat) // chans
        flat = [sum(flat[i * chans:(i + 1) * chans]) / chans for i in range(n)]
    return _finish(flat, rate, sample_rate, seconds=None), sample_rate


# ------------------------------------------------ browser recorder (Colab)
_BROWSER_JS = r"""
window._ling345 = async function(mode, ms, targetRate) {
  const panel = document.createElement('div');
  panel.style.cssText = 'font:15px/1.5 system-ui,sans-serif;padding:12px 14px;border:2px solid #3367d6;'
                      + 'border-radius:10px;display:inline-block;min-width:280px;background:#f6f9ff;color:#111';
  const line = document.createElement('div'); panel.appendChild(line);
  const slot = document.createElement('div'); slot.style.marginTop = '8px'; panel.appendChild(slot);
  document.body.appendChild(panel);
  const say = t => { line.textContent = t; };
  const sleep = t => new Promise(r => setTimeout(r, t));
  let blob;
  try {
    if (mode === 'file') {
      say('Choose a sound file from your device:');
      const inp = document.createElement('input');
      inp.type = 'file'; inp.accept = 'audio/*,video/*';
      slot.appendChild(inp);
      blob = await new Promise((res, rej) => {
        inp.onchange = () => inp.files && inp.files.length ? res(inp.files[0])
                                                          : rej(new Error('No file was chosen.'));
        setTimeout(() => rej(new Error('Timed out waiting for a file (5 minutes).')), 300000);
      });
      say('Reading ' + blob.name + ' ...');
    } else {
      if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia)
        throw new Error('This browser does not offer microphone access (no getUserMedia).');
      say('Click "Allow" if the browser asks to use your microphone...');
      const stream = await navigator.mediaDevices.getUserMedia({audio: {
        channelCount: 1, echoCancellation: false, noiseSuppression: false, autoGainControl: false}});
      for (const n of ['3', '2', '1']) { say('Get ready...  ' + n); await sleep(650); }
      const rec = new MediaRecorder(stream), chunks = [];
      rec.ondataavailable = e => { if (e.data && e.data.size) chunks.push(e.data); };
      const stopped = new Promise(r => { rec.onstop = r; });
      rec.start();
      say('*** RECORDING -- SPEAK NOW ***');
      await sleep(ms);
      rec.stop();
      stream.getTracks().forEach(t => t.stop());
      await stopped;
      if (!chunks.length) throw new Error('The microphone produced no audio at all.');
      blob = new Blob(chunks, {type: chunks[0].type || 'audio/webm'});
      say('Done. Listen back if you like:');
    }
    const bytes = await blob.arrayBuffer();
    const player = document.createElement('audio');
    player.controls = true; player.src = URL.createObjectURL(blob); slot.appendChild(player);

    // Decode + resample in the browser: no Python audio libraries needed.
    let buf = null, rate = targetRate;
    const OAC = window.OfflineAudioContext || window.webkitOfflineAudioContext;
    if (OAC) {
      try {                                    // decodeAudioData resamples to ctx rate
        buf = await new OAC(1, 1, targetRate).decodeAudioData(bytes.slice(0));
        rate = buf.sampleRate;
      } catch (e) { buf = null; }
    }
    if (!buf) {                                // fall back: native rate, Python resamples
      const AC = window.AudioContext || window.webkitAudioContext;
      const ac = new AC();
      buf = await ac.decodeAudioData(bytes.slice(0));
      rate = buf.sampleRate;
      if (ac.close) ac.close();
    }
    const n = buf.length, ch = buf.numberOfChannels, mono = new Float32Array(n);
    for (let c = 0; c < ch; c++) {
      const d = buf.getChannelData(c);
      for (let i = 0; i < n; i++) mono[i] += d[i] / ch;   // down-mix to mono
    }
    const pcm = new Int16Array(n);
    for (let i = 0; i < n; i++) {
      const s = Math.max(-1, Math.min(1, mono[i]));
      pcm[i] = s < 0 ? s * 32768 : s * 32767;
    }
    const u8 = new Uint8Array(pcm.buffer);
    let bin = '';
    for (let i = 0; i < u8.length; i += 8192)
      bin += String.fromCharCode.apply(null, u8.subarray(i, i + 8192));
    say((mode === 'file' ? 'Loaded ' : 'Recorded ') + (n / rate).toFixed(2) + ' seconds of audio.');
    return JSON.stringify({ok: true, sampleRate: rate, pcm: btoa(bin)});
  } catch (err) {
    const name = (err && err.name) || 'Error', msg = (err && err.message) || String(err);
    say('Could not get audio: ' + name + ' -- ' + msg);
    return JSON.stringify({ok: false, name: name, message: msg});
  }
};
"""

_BROWSER_HINTS = {
    "NotAllowedError": "The browser blocked the microphone. Click the camera/mic icon in the "
                       "address bar (or the popup) and choose Allow, then run the cell again.",
    "NotFoundError":   "No microphone was found. Plug one in, or use upload_audio().",
    "NotReadableError":"Another app (Zoom, Teams, Voice Memos) is using the microphone. "
                       "Quit it and run the cell again.",
    "SecurityError":   "The notebook page is not allowed to use the microphone.",
    "AbortError":      "The browser gave up on the microphone. Try running the cell again.",
}


def _browser_capture(mode, seconds, sample_rate):
    from IPython.display import Javascript, display
    from google.colab import output
    display(Javascript(_BROWSER_JS))
    call = '_ling345("%s", %d, %d)' % (mode, int(seconds * 1000), int(sample_rate))
    limit = 330 if mode == "file" else int(seconds) + 120
    try:
        reply = output.eval_js(call, timeout_sec=limit)
    except Exception as e:
        raise MicrophoneError("The browser never answered (%s).\n%s" % (type(e).__name__, _HELP))
    if not reply:
        raise MicrophoneError("The browser sent nothing back. Re-run this cell; if the notebook "
                              "was reloaded, run the setup cell again first.\n" + _HELP)
    info = json.loads(reply)
    if not info.get("ok"):
        hint = _BROWSER_HINTS.get(info.get("name")) or ("The browser reported: %s -- %s"
                                                       % (info.get("name"), info.get("message")))
        raise MicrophoneError("%s\n%s" % (hint, _HELP))
    samples = _b64_pcm16_to_floats(info["pcm"])
    if not samples:
        raise MicrophoneError("The recording came back empty.\n" + _HELP)
    return _finish(samples, info.get("sampleRate", sample_rate), sample_rate,
                   seconds=(seconds if mode == "mic" else None))


# ------------------------------------------- local microphone (sounddevice)
def _local_capture(seconds, sample_rate):
    try:
        import sounddevice as sd
    except ImportError:
        raise MicrophoneError(
            "Recording on your own computer needs one small extra package.\n"
            "Put this in a new cell, run it, then come back here:\n"
            "\n"
            "    %pip install sounddevice\n"
            "\n"
            "(On Linux you may also need:  !apt-get install -y libportaudio2 )\n"
            "Or skip the microphone entirely and use demo_audio(\"no\").\n" + _HELP)
    try:
        rate = int(sample_rate)
        try:
            sd.check_input_settings(samplerate=rate, channels=1, dtype="float32")
        except Exception:                       # device refuses 16 kHz -> use its own rate
            rate = int(sd.query_devices(kind="input")["default_samplerate"])
        print("Get ready...")
        print("*** RECORDING -- SPEAK NOW (%.1f seconds) ***" % seconds)
        buf = sd.rec(int(seconds * rate), samplerate=rate, channels=1, dtype="float32",
                     blocking=True)
        print("Done.")
    except Exception as e:
        raise MicrophoneError("The microphone could not be opened (%s).\n%s" % (e, _HELP))
    flat = [float(v[0]) for v in buf]
    if max((abs(v) for v in flat), default=0.0) < 1e-4:
        raise MicrophoneError(
            "The recording is completely silent.\n"
            "On a Mac this usually means the app running Jupyter has not been given\n"
            "microphone permission: System Settings > Privacy & Security > Microphone.\n"
            "On Windows: Settings > Privacy & security > Microphone.\n"
            "Turn it on, restart Jupyter, and try again." + _HELP)
    return _finish(flat, rate, sample_rate, seconds=seconds)


# ------------------------------------------------------------ upload paths
def upload_audio(sample_rate=SAMPLE_RATE):
    """Use a sound file from your computer or phone instead of the microphone."""
    if _in_colab():
        return _browser_capture("file", 0, sample_rate)
    try:
        import ipywidgets
        from IPython.display import display
    except ImportError:
        raise MicrophoneError("Please put a .wav file next to this notebook and run:\n"
                              "    samples = load_wav('myvoice.wav')")
    box = ipywidgets.FileUpload(accept=".wav", multiple=False, description="Choose .wav")
    display(box)
    print("Click the button above and pick a .wav file,")
    print("then run the next cell.")
    return box


def finish_upload(box, sample_rate=SAMPLE_RATE):
    """Turn the file chosen with upload_audio() into samples (local Jupyter)."""
    value = getattr(box, "value", None)
    if not value:
        raise MicrophoneError("No file has been chosen yet. Click the button above first.")
    if isinstance(value, dict):                      # ipywidgets 7 (dict keyed by filename)
        item = list(value.values())[0]
        data = item["content"]
    else:                                            # ipywidgets 8 (list/tuple of dicts)
        data = value[0]["content"]
    data = bytes(data)
    samples, _ = wav_bytes_to_samples(data, sample_rate)
    return samples


def load_wav(path, sample_rate=SAMPLE_RATE):
    """Read a .wav file from disk into samples."""
    with open(path, "rb") as f:
        return wav_bytes_to_samples(f.read(), sample_rate)[0]


# ------------------------------------------------------ trimming silence
def trim_silence(samples, threshold=0.10):
    """Throw away the quiet run at each end of a recording.

    Silence is mostly microphone noise, and noise crosses zero constantly,
    so leaving it in swamps the eight numbers features() cares about.
    A sample counts as "sound" if it reaches `threshold` of the loudest one.
    """
    if not samples:
        return list(samples)
    peak = max(abs(v) for v in samples)
    if peak < 1e-6:
        return list(samples)
    loud = [i for i, v in enumerate(samples) if abs(v) >= threshold * peak]
    if not loud:
        return list(samples)
    return list(samples[loud[0]:loud[-1] + 1])


# --------------------------------------------------------------- stand-in
def demo_audio(word="no", seconds=1.0, sample_rate=SAMPLE_RATE):
    """A synthetic stand-in so the lesson works with no microphone at all.
    'no' = voiced nasal onset (few zero crossings); 'go' = noisy burst then vowel."""
    import random
    random.seed(1 if word == "go" else 0)   # a fixed seed: the same word every time
    n, out = int(seconds * sample_rate), []
    onset = int(0.18 * n)
    for i in range(n):
        t = i / float(sample_rate)
        env = min(1.0, i / (0.02 * sample_rate)) * min(1.0, (n - i) / (0.05 * sample_rate))
        if word == "go" and i < int(0.03 * n):
            s = random.uniform(-1, 1) * 0.6                      # burst: high ZCR
        elif i < onset and word == "no":
            s = 0.5 * math.sin(2 * math.pi * 250 * t) + 0.2 * math.sin(2 * math.pi * 500 * t)
        else:                                                    # the vowel
            s = (0.6 * math.sin(2 * math.pi * 130 * t)
                 + 0.3 * math.sin(2 * math.pi * 600 * t)
                 + 0.15 * math.sin(2 * math.pi * 1000 * t)
                 + 0.02 * random.uniform(-1, 1))
        out.append(s * env * 0.8)
    return _finish(out, sample_rate, sample_rate, seconds=seconds)


# ------------------------------------------------------------------ record
def record(seconds=2.0, sample_rate=SAMPLE_RATE, source="auto"):
    """Record `seconds` of your voice and return a list of floats (-1.0 .. 1.0).

    seconds     : how long to record
    sample_rate : samples per second in the result (default 16000)
    source      : "auto" (default), "mic", or "upload"
    """
    seconds = float(seconds)
    sample_rate = int(sample_rate)
    if source == "upload":
        return upload_audio(sample_rate)
    if _in_colab():
        return _browser_capture("mic", seconds, sample_rate)
    return _local_capture(seconds, sample_rate)


print("Audio helper ready.")
print('  record(2.0)          record from your microphone')
print('  demo_audio("no")     a stand-in sound, no microphone needed')
print('  upload_audio()       use a sound file from your computer')
print('  trim_silence(x)      cut the quiet ends off a recording')

### Way 1: a word the computer makes up

`demo_audio("no")` and `demo_audio("go")` build a second of fake speech: a hum or a burst followed by a vowel. Listen, then hand them to the recognizer you wrote. Same `recognize()` as Exercise 3, now running on 16000 numbers instead of 24.

In [ ]:
from IPython.display import Audio, display

fake_no = demo_audio("no")     # a hum, then a vowel
fake_go = demo_audio("go")     # a burst, then the same vowel

print('this one is meant to sound like "no":')
display(Audio(fake_no, rate=16000))
print('this one is meant to sound like "go":')
display(Audio(fake_go, rate=16000))

print()
print("These are 16000 numbers each, not 24, but features() does not care:")
print("no-ish ->", features(fake_no))
print("go-ish ->", features(fake_go))

if exercise_3_done():
    print()
    print("YOUR recognizer, on sound it has never seen:")
    print("  the no-ish recording ->", recognize(fake_no, model))
    print("  the go-ish recording ->", recognize(fake_go, model))

### Trimming the silence

Real recordings start and end with silence, and silence is mostly microphone noise, which crosses zero constantly. Left in, it swamps the eight numbers that matter.

So cut the quiet ends off before calling `features()`. Finding where speech starts and stops has a name, **endpoint detection**, and the classic version of it uses exactly what we have here: energy and zero-crossing rate.

In [ ]:
import random
random.seed(7)

# a real recording of "no": a second of quiet room, the word, quiet room again
quiet = [random.uniform(-0.01, 0.01) for _ in range(4000)]     # microphone noise
noisy = quiet + demo_audio("no") + quiet

clean = trim_silence(noisy)

print("before trimming:", len(noisy), "samples")
print("after trimming: ", len(clean), "samples")
print()
print("features with the quiet ends left in:", features(noisy))
print("features with the quiet ends cut off:", features(clean))
print()
if exercise_3_done():
    print("the recognizer, silence left in:", recognize(noisy, model))
    print("the recognizer, silence cut off:", recognize(clean, model))
    print()
    print("Room noise is quiet, but it crosses zero constantly, so the")
    print("silent ends look like a burst. Left in, they talk the recognizer")
    print("into hearing a 'g' in a word that has none.")

### Way 2: your own voice

Say **no** into the microphone once, right after the countdown. Do this three times, then the same for **go**. That is your training data.

Leave a moment of silence at each end. Averaging three takes of a word is a real, if tiny, version of training a speech model.

If the microphone does not work, skip these cells. Nothing below depends on them.

In [ ]:
# Say the word once, clearly, right after the countdown.
# Change the word or the number of takes if you like.

my_takes = {"no": [], "go": []}

try:
    for word in ["no", "go"]:
        for take in range(3):
            print()
            print('=== say "%s"  (take %d of 3) ===' % (word, take + 1))
            sound = record(1.5)             # 1.5 seconds from your microphone
            my_takes[word].append(trim_silence(sound))
except MicrophoneError as problem:
    print(problem)                          # a plain explanation, not a crash

print()
for word in my_takes:
    print(word, "->", len(my_takes[word]), "takes,",
          [len(t) for t in my_takes[word]], "samples each")

In [ ]:
# TRAINING, for real this time: average the three takes of each word.

my_model = {}
for word in my_takes:
    each = [features(take) for take in my_takes[word]]          # 8 numbers per take
    if not each:                                                # nothing recorded for this word
        continue
    my_model[word] = [round(sum(f[i] for f in each) / len(each), 3)
                      for i in range(8)]                        # averaged, slice by slice

if len(my_model) < 2:
    print("There are not two words to tell apart yet.")
    print("Run the recording cell above first, or skip ahead to Way 3.")
else:
    for word in my_model:
        print(word, "->", my_model[word])
    print()
    print("Your two words differ most where these two lists differ most.")

### Now say one of them again

Run the cell, say either **no** or **go**, and watch the two distances.

No worries if this simplified model makes mistakes! Eight zero-crossing numbers is an absurdly small description of a word: no pitch, no vowel quality, no duration. The accuracy will improve if you go back and record more takes.

In [ ]:
# Say either "no" or "go" after the countdown.

my_model = globals().get("my_model", {})     # empty if you skipped the cells above

try:
    test_sound = trim_silence(record(1.5))
except MicrophoneError as problem:
    test_sound = None
    print(problem)

if test_sound and len(my_model) == 2:
    guess = recognize(test_sound, my_model)            # YOUR function, your model
    print()
    print("the recognizer says:", guess)
    print()
    for word in my_model:
        print("  distance to %-3s = %.4f"
              % (word, distance(features(test_sound), my_model[word])))
    print()
    # thin the recording out before drawing it: 16000 dots is not a picture
    show_wave(test_sound[::max(1, len(test_sound) // 400)], "what you just said")
elif test_sound:
    print("Record your training takes further up before testing.")

### Way 3: upload a sound file

Record a voice memo on your phone, get the file onto this computer, and load it here.

In Colab any format works, since the browser does the decoding. In local Jupyter use a `.wav`; most phones record `.m4a`, so you may need to convert it.

In [ ]:
# In Colab this opens a file picker and hands you the samples directly.
# In local Jupyter it shows an upload button; run the NEXT cell afterwards.

try:
    picked = upload_audio()
except MicrophoneError as problem:
    picked = None
    print(problem)

if picked is None:
    pass                                    # the message above says what to do
elif not isinstance(picked, list):
    print()
    print("Now run the next cell.")
else:
    uploaded = trim_silence(picked)         # Colab hands the samples straight back
    print(len(uploaded), "samples")
    print("features ->", features(uploaded))
    if exercise_3_done():
        print("the recognizer says:", recognize(uploaded, model))

In [ ]:
# Local Jupyter only: run this AFTER you have chosen a file above.
# In Colab the cell above already did everything.
if picked is None or isinstance(picked, list):
    print("Nothing to do here -- see the cell above.")
elif not getattr(picked, "value", None):
    print("No file chosen yet. Click the button in the cell above,")
    print("pick a .wav file, then run this cell again.")
else:
    uploaded = trim_silence(finish_upload(picked))
    print(len(uploaded), "samples")
    print("features ->", features(uploaded))
    if exercise_3_done():
        print("the recognizer says:", recognize(uploaded, model))

## Your report

In [ ]:
report()

## How to hand this in

**Run everything one last time.** **Runtime &rarr; Restart session and run all**. Only cells you have actually run show their output, and only what is on the screen ends up in the PDF.

**Make the PDF.** **File &rarr; Print**, set *Destination* to **Save as PDF**, then **Save**. That is the normal route; Colab has no other PDF export. (In local Jupyter: **File &rarr; Save and Export Notebook As&hellip; &rarr; HTML**, open the `.html` in a browser, then print to PDF. Do not pick "PDF" directly, which needs LaTeX and will fail.)

**Upload the PDF to Canvas.** To keep the working notebook too, use **File &rarr; Download &rarr; Download .ipynb**.

If something did not work &mdash; the microphone, a cell that will not run, anything &mdash; say so when you submit. A broken tool is my problem, not your evening.